# Inference Best Practices

Efficient inference combines the model's evaluation behavior with a gradient-disabling context, consistent device placement, and an appropriate batch size.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Inference device: {device}")

## Prepare a Model and Test Data

Dropout makes the distinction between training and evaluation behavior visible. The generated dataset keeps this demonstration self-contained.

In [ ]:
model = nn.Sequential(
    nn.Linear(16, 32),
    nn.ReLU(),
    nn.Dropout(0.25),
    nn.Linear(32, 4),
).to(device)

features = torch.randn(128, 16)
targets = torch.randint(0, 4, (128,))
test_dataset = TensorDataset(features, targets)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## `model.eval()` Controls Layer Behavior

Evaluation mode changes modules such as Dropout and BatchNorm. It does **not** disable gradient tracking, so it is used together with `no_grad()` or `inference_mode()`.

In [ ]:
sample = features[:1].to(device)

model.train()
training_outputs = [model(sample) for _ in range(3)]

model.eval()
evaluation_outputs = [model(sample) for _ in range(3)]

print("Training outputs equal:", torch.equal(training_outputs[0], training_outputs[1]))
print("Evaluation outputs equal:", torch.equal(evaluation_outputs[0], evaluation_outputs[1]))
print("Gradients tracked after eval():", evaluation_outputs[0].requires_grad)

## `no_grad()` and `inference_mode()`

Both contexts prevent an autograd graph from being built. `inference_mode()` also removes additional bookkeeping and is preferred when the resulting tensors are used only for inference. `no_grad()` is more flexible when its output must later participate in autograd-enabled work.

In [ ]:
model.eval()

with torch.no_grad():
    no_grad_output = model(sample)

with torch.inference_mode():
    inference_output = model(sample)

print("no_grad tracks gradients:", no_grad_output.requires_grad)
print("inference_mode tracks gradients:", inference_output.requires_grad)
print("Predictions match:", torch.allclose(no_grad_output, inference_output))

## Benchmark Representative Workloads

Benchmark after warmup and over many iterations. CUDA work is asynchronous, so synchronize before reading the timer. Results from this small example illustrate measurement technique rather than guaranteeing that one context is always faster.

In [ ]:
from time import perf_counter

def synchronize_device():
    if device.type == "cuda":
        torch.cuda.synchronize()

def benchmark(context_factory, iterations=200):
    for _ in range(20):
        with context_factory():
            model(sample)
    synchronize_device()
    start = perf_counter()
    for _ in range(iterations):
        with context_factory():
            model(sample)
    synchronize_device()
    return perf_counter() - start

print(f"no_grad:       {benchmark(torch.no_grad):.6f} seconds")
print(f"inference_mode: {benchmark(torch.inference_mode):.6f} seconds")

## Single-Sample Inference

Single requests can prioritize latency. Add a batch dimension when the model expects batched input and keep the model and input on the same device.

In [ ]:
single_sample = features[0].unsqueeze(0).to(device)

with torch.inference_mode():
    single_logits = model(single_sample)
    single_prediction = single_logits.argmax(dim=1)

print("Single prediction:", single_prediction.item())

## Batch Inference

Batching usually improves throughput. Move each batch to the selected device, retain only what is needed, and transfer stored predictions back to CPU so accelerator memory can be released.

In [ ]:
all_predictions = []
correct = 0

model.eval()
with torch.inference_mode():
    for batch_features, batch_targets in test_loader:
        batch_features = batch_features.to(device)
        batch_targets = batch_targets.to(device)
        batch_predictions = model(batch_features).argmax(dim=1)
        correct += (batch_predictions == batch_targets).sum().item()
        all_predictions.append(batch_predictions.cpu())

all_predictions = torch.cat(all_predictions)
print("Stored predictions:", all_predictions.shape)
print(f"Accuracy: {100.0 * correct / len(test_dataset):.2f}%")

## Choosing a Batch Size

Larger batches can improve throughput but consume more memory. Interactive applications may prefer small batches for lower latency, while offline evaluation can use the largest stable batch size. Benchmark representative workloads after warmup and synchronize CUDA around timing measurements.